# 00 - Data Audit

Load all panels, verify structure, and confirm treatment coding before analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

## FARS panel structure

In [ ]:
print(f"Years: {sorted(fars['year'].dropna().unique())}")
print(f"\nTreatment breakdown:")
cohorts = (
    leg[leg['retail_sales_year'].notna()]
    .groupby('retail_sales_year')['state'].apply(list)
    .sort_index()
)
for yr, states in cohorts.items():
    print(f"  {int(yr)}: {', '.join(states)}")
print(f"\nNever treated: {leg[leg['retail_sales_year'].isna()]['state'].nunique()} states")

## Check for missing outcomes

In [ ]:
outcome_cols = [c for c in fars.columns if 'fatalities' in c or 'per_100k' in c]
print("Missing values in outcome columns:")
print(fars[outcome_cols].isnull().sum())

## CDC panel (if available)

In [ ]:
try:
    cdc = pd.read_parquet(DATA_DIR / CDC_FILE)
    print(f"CDC panel: {cdc.shape}")
    print(cdc[["state","year","overdose_deaths_per_100k"]].describe())
except FileNotFoundError:
    print("CDC panel not yet built. Run: python src/build_cdc_panel.py")

## NYC 311 panel (if available)

In [ ]:
try:
    nyc_311 = pd.read_parquet(DATA_DIR / "nyc_311_zip_month.parquet")
disp     = pd.read_parquet(DATA_DIR / "nyc_dispensary_events.parquet")
print(f"311 panel: {nyc_311.shape}  |  Zips: {nyc_311['zip'].nunique()}")
print(f"Dispensaries: {len(disp)}")
except FileNotFoundError as e:
    print(f"NYC 311 data not yet built: {e}")